# Здесь будем работать с моделью Vision Transfromer

## Рассмотрим для начала простейшую архитектуру модели без предобучения

In [23]:
import os
import random
from pathlib import Path
from urllib.request import urlretrieve
import requests
import dagshub
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ViT_B_16_Weights
from tqdm.auto import tqdm

In [24]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)


seed_everything(42)

In [25]:
class CFG:
    seed = 42
    image_col = "image_link"
    target_col = "price"
    cache_dir = "../data/images_cache"
    img_size = 224
    batch_size = 16
    num_workers = 0
    epochs = 5
    lr = 1e-4
    weight_decay = 1e-5
    val_size = 0.2
    pretrained = False # фиксируем без предобучения
    freeze_backbone = False # не замораживаем слои
    dropout = 0.3 # фиксируем дропаут
    hidden_size = 256
    artifacts_dir = "model_outputs/ViT-v0"
    model_type = 'ViT-v0'
    model_name = "ViT-v0.pt"
    experiment_name = "amazon_smart_pricing"
    run_name = "ViT-v0-no-pretrain" # название на сайте

In [26]:
dagshub.init(
    repo_owner="Dezurn",
    repo_name="Deep-Learning",
    mlflow=True,
)

mlflow.set_experiment(CFG.experiment_name)

Initialized MLflow to track repo "Dezurn/Deep-Learning"

Repository Dezurn/Deep-Learning initialized!

<Experiment: artifact_location='mlflow-artifacts:/99b0aadd413f4a00bf17d371a1a7a7aa', creation_time=1781452541346, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781452541346, lifecycle_stage='active', name='amazon_smart_pricing', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [27]:
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")
train_df.head()

,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


In [28]:
num = 5000
work_df = train_df.sample(num, random_state=CFG.seed).reset_index(drop=True)
val_data = work_df.iloc[: int(len(work_df) * CFG.val_size)].reset_index(drop=True)
train_data = work_df.iloc[int(len(work_df) * CFG.val_size) :].reset_index(drop=True)

In [29]:
len(val_data), len(train_data)

(1000, 4000)

In [30]:
from urllib.request import urlretrieve
from pathlib import Path
from tqdm.auto import tqdm
def download_images_simple(df):
    Path(CFG.cache_dir).mkdir(parents=True, exist_ok=True)

    image_paths = []

    for url in tqdm(df[CFG.image_col], desc="Downloading images"):
        filename = str(url).split("/")[-1].split("?")[0]
        image_path = f"{CFG.cache_dir}/{filename}"

        try:
            if not Path(image_path).exists():
                urlretrieve(url, image_path)

            image_paths.append(image_path)

        except Exception:
            image_paths.append(None)

    df = df.copy()
    df["image_path"] = image_paths
    df = df.dropna(subset=["image_path"]).reset_index(drop=True)

    return df

In [ ]:
import requests
from pathlib import Path
from tqdm.auto import tqdm

def download_images_simple(df):
    Path(CFG.cache_dir).mkdir(parents=True, exist_ok=True)

    image_paths = []
    errors = []

    for url in tqdm(df[CFG.image_col], desc="Downloading images"):
        filename = str(url).split("/")[-1].split("?")[0]
        image_path = Path(CFG.cache_dir) / filename

        try:
            if not image_path.exists():
                response = requests.get(url, headers=headers, timeout=20)
                response.raise_for_status()

                with open(image_path, "wb") as f:
                    f.write(response.content)

            image_paths.append(str(image_path))

        except Exception as e:
            image_paths.append(None)
            errors.append((url, str(e)))

    df = df.copy()
    df["image_path"] = image_paths

    print("Всего строк:", len(df))
    print("Скачано/найдено картинок:", df["image_path"].notna().sum())
    print("Ошибок:", len(errors))

    if errors:
        print("Пример ошибки:")
        print(errors[0])

    df = df.dropna(subset=["image_path"]).reset_index(drop=True)

    return df

In [32]:
train_data = download_images_simple(train_data)
val_data = download_images_simple(val_data)

Всего строк: 4000
Скачано/найдено картинок: 4000
Ошибок: 0


Всего строк: 1000
Скачано/найдено картинок: 1000
Ошибок: 0


In [33]:
train_data.head(), val_data.head()

(   sample_id                                    catalog_content  \
 0     156787  Item Name: 'Rich JW Allen Snow Queen Icing Bas...   
 1     225070  Item Name: Dandies Vegan Marshmallows, Vanilla...   
 2      46913  Item Name: Canels Gum Box Original 60 Count\r\...   
 3     298119  Item Name: HERSHEY'S Miniatures Assorted Choco...   
 4      68001  Item Name: Pink Lemonade Licorice Sour Sticks ...   
 
                                           image_link    price  \
 0  https://m.media-amazon.com/images/I/71mDwF1ANo...  113.310   
 1  https://m.media-amazon.com/images/I/71hAUfz93S...    5.490   
 2  https://m.media-amazon.com/images/I/91DIhiMcqI...    4.035   
 3  https://m.media-amazon.com/images/I/51iYRznRks...   20.790   
 4  https://m.media-amazon.com/images/I/91hRyLkNi+...   18.990   
 
                              image_path  
 0  ..\data\images_cache\71mDwF1ANoL.jpg  
 1  ..\data\images_cache\71hAUfz93SL.jpg  
 2  ..\data\images_cache\91DIhiMcqIL.jpg  
 3  ..\data\images_c

In [34]:
class ImagePriceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        price = np.log1p(row[CFG.target_col]).astype("float32")

        return image, torch.tensor(price, dtype=torch.float32)

In [35]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
train_transform = T.Compose([
    T.Resize(256), # меняем размер
    T.RandomCrop(CFG.img_size), # вырезаем случайную область указанного размера
    T.RandomHorizontalFlip(), # зеркальное отражение по горизонтали
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD) # нормализация
])

valid_transform = T.Compose([ # на валидации нельзя делать никаких преобразований (кроме изменения размера),
                               # ведь мы должны проверить модель на реальных картинках
    T.Resize(256),
    T.CenterCrop(CFG.img_size),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

In [36]:
train_dataset = ImagePriceDataset(train_data, transform=train_transform)
val_dataset = ImagePriceDataset(val_data, transform=valid_transform)

train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)

val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Дальше сама модель.

Функции почти не менял, только где-то другие параметры

In [38]:
class ViTPriceRegressor(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=True, hidden_size=256, dropout=0.3):
        super().__init__()
        if pretrained:
            weights = ViT_B_16_Weights.DEFAULT
            self.model = models.vit_b_16(weights=weights)
        else:
            self.model = models.vit_b_16(weights=None)
        #для заморозки части модели
        if freeze_backbone:
            for param in self.model.parameters():
                param.requires_grad = False
        #меняем классификатор
        in_features = self.model.heads.head.in_features
        #слои
        self.model.heads.head = nn.Sequential(
            nn.Linear(in_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(1)

In [39]:
model_vit = ViTPriceRegressor(
    pretrained=CFG.pretrained,
    freeze_backbone=CFG.freeze_backbone,
    hidden_size=CFG.hidden_size,
    dropout=CFG.dropout
).to(device)

model_vit

ViTPriceRegressor(
  (model): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): Encoder

In [40]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    filter(lambda param: param.requires_grad, model_vit.parameters()),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

In [41]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0

    for images, targets in tqdm(loader, desc="Train", leave=False):
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)

    return epoch_loss

In [42]:
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0

    preds = []
    targets_all = []

    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Valid", leave=False):
            images = images.to(device)
            targets = targets.to(device)

            outputs = model(images)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * images.size(0)

            preds.extend(outputs.detach().cpu().numpy())
            targets_all.extend(targets.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    preds = np.array(preds)
    targets_all = np.array(targets_all)

    preds_price = np.expm1(preds)
    targets_price = np.expm1(targets_all)

    preds_price = np.clip(preds_price, 0, None)

    mae = mean_absolute_error(targets_price, preds_price)
    mse = mean_squared_error(targets_price, preds_price)
    rmse = np.sqrt(mse)
    r2 = r2_score(targets_price, preds_price)

    metrics = {
        "val_loss": epoch_loss,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
    }

    return metrics

In [ ]:
def fit_model(model, train_loader, val_loader, optimizer, scheduler, criterion, device):
    best_rmse = float("inf")
    history = []

    os.makedirs(CFG.artifacts_dir, exist_ok=True)
    best_model_path = os.path.join(CFG.artifacts_dir, CFG.model_name)
    history_path = os.path.join(CFG.artifacts_dir, "vit_history.csv")

    mlflow.log_params(
        {
            "model": CFG.model_type,
            "pretrained": CFG.pretrained,
            "frozen_backbone": CFG.freeze_backbone,
            "hidden_size": CFG.hidden_size,
            "dropout": CFG.dropout,
            "target": "log1p(price)",
            "img_size": CFG.img_size,
            "batch_size": CFG.batch_size,
            "epochs": CFG.epochs,
            "lr": CFG.lr,
            "weight_decay": CFG.weight_decay,
            "optimizer": "AdamW",
            "loss": "MSELoss",
            "scheduler": "ReduceLROnPlateau",
            "val_size": CFG.val_size,
            "seed": CFG.seed,
        }
    )

    for epoch in range(1, CFG.epochs + 1):
        print(f"\nEpoch {epoch}/{CFG.epochs}")

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
        )

        val_metrics = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device,
        )

        scheduler.step(val_metrics["val_loss"])

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["val_loss"],
            "mae": val_metrics["mae"],
            "rmse": val_metrics["rmse"],
            "r2": val_metrics["r2"],
        }

        history.append(row)

        mlflow.log_metrics(row, step=epoch)

        print(
            f"train_loss: {train_loss:.4f} | "
            f"val_loss: {val_metrics['val_loss']:.4f} | "
            f"MAE: {val_metrics['mae']:.2f} | "
            f"RMSE: {val_metrics['rmse']:.2f} | "
            f"R2: {val_metrics['r2']:.4f}"
        )

        if val_metrics["rmse"] < best_rmse:
            best_rmse = val_metrics["rmse"]

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "epoch": epoch,
                    "best_rmse": best_rmse,
                },
                best_model_path,
            )

            print(f"Best model saved: {best_model_path}")

    history = pd.DataFrame(history)
    history.to_csv(history_path, index=False)
    
    best_checkpoint = torch.load(
    best_model_path,
    map_location=device,
    weights_only=False)
    model.load_state_dict(best_checkpoint["model_state_dict"])

    mlflow.log_metric("best_rmse", best_rmse)
    mlflow.log_artifact(best_model_path, artifact_path="model_checkpoint")
    mlflow.log_artifact(history_path, artifact_path="history")
    mlflow.pytorch.log_model(model, artifact_path="model")

    return model, history

In [44]:
with mlflow.start_run(run_name=CFG.run_name):
    model_vit, history_vit = fit_model(
        model=model_vit,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=device,
    )


Epoch 1/5


Train:   0%|          | 0/250 [00:00<?, ?it/s]

Valid:   0%|          | 0/63 [00:00<?, ?it/s]

train_loss: 1.0763 | val_loss: 0.8494 | MAE: 15.76 | RMSE: 31.32 | R2: -0.0781
Best model saved: model_outputs/ViT-v0\ViT-v0.pt

Epoch 2/5


Train:   0%|          | 0/250 [00:00<?, ?it/s]

...

Train:   0%|          | 0/250 [00:00<?, ?it/s]

Valid:   0%|          | 0/63 [00:00<?, ?it/s]

train_loss: 0.9921 | val_loss: 0.8478 | MAE: 15.84 | RMSE: 30.39 | R2: -0.0149
Best model saved: model_outputs/ViT-v0\ViT-v0.pt


2026/06/15 18:13:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/15 18:13:34 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/15 18:13:34 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.12.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/15 18:13:41 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.27.0+cu126) contains a local version

🏃 View run ViT-v0-no-pretrain at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1/runs/29dae8304d43495ca2e641235723f38f
🧪 View experiment at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1


Попробуем еще поиграться с дропаутом и бачсайзом, сильного переобучения не наблюдается.

Модель и так довольно долго обучается, но попробуем снизить дропаут и увеличить бачсайз. И обучить на более мощной видюхе

In [45]:
CFG.batch_size = 32
CFG.dropout = 0.15

In [46]:
class ViTPriceRegressor(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=True, hidden_size=256, dropout=0.3):
        super().__init__()
        if pretrained:
            weights = ViT_B_16_Weights.DEFAULT
            self.model = models.vit_b_16(weights=weights)
        else:
            self.model = models.vit_b_16(weights=None)
        #для заморозки части модели
        if freeze_backbone:
            for param in self.model.parameters():
                param.requires_grad = False
        #меняем классификатор
        in_features = self.model.heads.head.in_features
        #слои
        self.model.heads.head = nn.Sequential(
            nn.Linear(in_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(1)

In [ ]:
model_vit = ViTPriceRegressor(
    pretrained=CFG.pretrained,
    freeze_backbone=CFG.freeze_backbone,
    hidden_size=CFG.hidden_size,
    dropout=CFG.dropout
).to(device)

model_vit

ViTPriceRegressor(
  (model): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): Encoder

In [48]:
with mlflow.start_run(run_name=CFG.run_name):
    model_vit, history_vit = fit_model(
        model=model_vit,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=device,
    )


Epoch 1/5


Train:   0%|          | 0/250 [00:00<?, ?it/s]

Valid:   0%|          | 0/63 [00:00<?, ?it/s]

train_loss: 8.7919 | val_loss: 8.9214 | MAE: 23.15 | RMSE: 38.02 | R2: -0.5891
Best model saved: model_outputs/ViT-v0\ViT-v0.pt

Epoch 2/5


Train:   0%|          | 0/250 [00:00<?, ?it/s]

...

Train:   0%|          | 0/250 [00:00<?, ?it/s]

Valid:   0%|          | 0/63 [00:00<?, ?it/s]

train_loss: 8.8007 | val_loss: 8.9214 | MAE: 23.15 | RMSE: 38.02 | R2: -0.5891


2026/06/15 19:07:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/15 19:07:20 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/15 19:07:20 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.12.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/15 19:07:25 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.27.0+cu126) contains a local version

🏃 View run ViT-v0-no-pretrain at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1/runs/4e1e345eb50a470a8e2560b87b94f5ba
🧪 View experiment at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1


Можно сказать, что первоначально мы подобрали достаточно хороший размер батча и дропаут. С батчем 32 модель работает очень долго, дальше протестируем предобученную модель на прошлых параметрах и посмотрим на результаты.

## Перейдем к предобученной модели

In [105]:
class CFG:
    seed = 42
    image_col = "image_link"
    target_col = "price"
    cache_dir = "../data/images_cache"
    img_size = 224
    batch_size = 16
    num_workers = 0
    epochs = 3 # понижаем кол-во епох
    lr = 1e-4
    weight_decay = 1e-5
    val_size = 0.2
    pretrained = True # фиксируем предобучение
    freeze_backbone = True # замораживаем слои
    dropout = 0.3 # фиксируем дропаут
    hidden_size = 256
    artifacts_dir = "model_outputs/ViT-v1"
    model_type = 'ViT-v1'
    model_name = "ViT-v1.pt"
    experiment_name = "amazon_smart_pricing"
    run_name = "ViT-v1-pretrain" # название на сайте

In [106]:
class ViTPriceRegressor(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=True, hidden_size=256, dropout=0.3):
        super().__init__()
        if pretrained:
            weights = ViT_B_16_Weights.DEFAULT
            self.model = models.vit_b_16(weights=weights)
        else:
            self.model = models.vit_b_16(weights=None)
        #для заморозки части модели
        if freeze_backbone:
            for param in self.model.parameters():
                param.requires_grad = False
        #меняем классификатор
        in_features = self.model.heads.head.in_features
        #слои
        self.model.heads.head = nn.Sequential(
            nn.Linear(in_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(1)

In [107]:
model_vit = ViTPriceRegressor(
    pretrained=CFG.pretrained,
    freeze_backbone=CFG.freeze_backbone,
    hidden_size=CFG.hidden_size,
    dropout=CFG.dropout
).to(device)

model_vit

ViTPriceRegressor(
  (model): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): Encoder

In [108]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    filter(lambda param: param.requires_grad, model_vit.parameters()),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

In [109]:
with mlflow.start_run(run_name=CFG.run_name):
    model_vit, history_vit = fit_model(
        model=model_vit,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=device,
    )

KeyboardInterrupt: 

Замораживаем основную часть и используем предобученные веса из ImageNet

Вывод:

## Обучаем всю модель

In [119]:
class CFG:
    seed = 42
    image_col = "image_link"
    target_col = "price"
    cache_dir = "../data/images_cache"
    img_size = 224
    batch_size = 4 # иначе будет очень долго работать
    num_workers = 0
    epochs = 3 # особо много эппох тут не нужно, оставим 3, чтобы не работало так долго
    lr = 1e-5 #уменьшил еще lr
    weight_decay = 1e-5
    val_size = 0.2
    pretrained = True # фиксируем предобучение
    freeze_backbone = False # замораживаем слои
    dropout = 0.3 # фиксируем дропаут
    hidden_size = 256
    artifacts_dir = "model_outputs/ViT-v2"
    model_type = 'ViT-v2'
    model_name = "ViT-v2.pt"
    experiment_name = "amazon_smart_pricing"
    run_name = "ViT-v2-full-pretrain" # название на сайте

In [120]:
train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)
val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

Дообучаем регрессию по V1

In [121]:
model_vit = ViTPriceRegressor(
    pretrained=CFG.pretrained,
    freeze_backbone=CFG.freeze_backbone,
    hidden_size=CFG.hidden_size,
    dropout=CFG.dropout
).to(device)

model_vit

ViTPriceRegressor(
  (model): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): Encoder

In [115]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    filter(lambda param: param.requires_grad, model_vit.parameters()),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

In [104]:
with mlflow.start_run(run_name=CFG.run_name):
    model_vit, history_vit = fit_model(
        model=model_vit,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=device,
    )


Epoch 1/3


🏃 View run ViT-v2-pretrain at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1/runs/00b324b1703a4113918525e24265b9d2
🧪 View experiment at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1


KeyboardInterrupt: 

Вывод:

## Финальная версия с большим числом нейронов

In [ ]:
class CFG:
    seed = 42
    image_col = "image_link"
    target_col = "price"
    cache_dir = "../data/images_cache"
    img_size = 224
    batch_size = 4
    num_workers = 0
    epochs = 2
    lr = 1e-5 #уменьшил еще lr
    weight_decay = 1e-5
    val_size = 0.2
    pretrained = True # фиксируем предобучение
    freeze_backbone = False # замораживаем слои
    dropout = 0.35 # поднимаем дропаут
    hidden_size = 512 # добавляем нейроны
    artifacts_dir = "model_outputs/ViT-v3"
    model_type = 'ViT-v3'
    model_name = "ViT-v3.pt"
    experiment_name = "amazon_smart_pricing"
    run_name = "ViT-v3-final" # название на сайте

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)
val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

In [117]:
model_vit = ViTPriceRegressor(
    pretrained=CFG.pretrained,
    freeze_backbone=CFG.freeze_backbone,
    hidden_size=CFG.hidden_size,
    dropout=CFG.dropout
).to(device)

model_vit

ViTPriceRegressor(
  (model): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): Encoder

In [118]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    filter(lambda param: param.requires_grad, model_vit.parameters()),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

In [ ]:
with mlflow.start_run(run_name=CFG.run_name):
    model_vit, history_vit = fit_model(
        model=model_vit,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=device,
    )

Вывод:

**Итоговый Вывод:**